# Stage 2 — Data Cleaning & Wrangling

The Stage 1 profile showed zero nulls and zero duplicates, so the real work here is **type
discipline, business-rule validation, and feature derivation** — not repair.

Every transformation is logged with rows-affected and a rationale. Nothing is dropped silently.

**Gate:** null strategy per column · duplicates justified · dtypes correct · business rules pass ·
cleaning log exists · processed data saved.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw" / "Superstore.csv"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RAW, encoding="latin-1")
n_start = len(df)
print(f"Loaded {n_start:,} rows x {df.shape[1]} columns from the immutable raw copy.")

# Cleaning log — every step appends a row.
log: list[dict] = []


def record(step: str, rows_before: int, rows_after: int, rationale: str) -> None:
    log.append({
        "step": step,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_affected": rows_before - rows_after,
        "rationale": rationale,
    })

Loaded 9,994 rows x 21 columns from the immutable raw copy.


## 2.1 Missing values — verify the claim

The dataset is *reported* to have zero nulls. We test rather than trust, and we check for
**disguised** missingness too: empty strings, whitespace-only cells, and sentinel values that
`isnull()` will happily report as present.

In [2]:
null_counts = df.isnull().sum()
obj_cols = df.select_dtypes(include="object").columns
blank = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in obj_cols}
sentinels = {c: int(df[c].astype(str).str.strip().str.lower()
                    .isin({"na", "n/a", "null", "none", "unknown", "-", "?"}).sum())
             for c in obj_cols}

print(f"Explicit nulls (isnull)        : {null_counts.sum()}")
print(f"Empty / whitespace-only strings: {sum(blank.values())}")
print(f"Sentinel values ('NA','?',...) : {sum(sentinels.values())}")
print()
print("STRATEGY: no imputation or deletion required — there is nothing missing, disguised or")
print("otherwise. Had any column shown missingness we would classify it MCAR/MAR/MNAR and")
print("document a per-column strategy. Recorded as a no-op so the log is honest about it.")

record("Missing-value handling", n_start, n_start,
       "No nulls, blanks, or sentinels found. No imputation or deletion applied (no-op).")

Explicit nulls (isnull)        : 0
Empty / whitespace-only strings: 0
Sentinel values ('NA','?',...) : 0

STRATEGY: no imputation or deletion required — there is nothing missing, disguised or
otherwise. Had any column shown missingness we would classify it MCAR/MAR/MNAR and
document a per-column strategy. Recorded as a no-op so the log is honest about it.


C:\Users\HP ENVY\AppData\Local\Temp\ipykernel_14312\943366151.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns


## 2.2 Duplicates — exact vs. legitimate business repeats

This is the trap in this dataset. `Order ID` repeats by design (one order holds many product
lines), and even `Order ID + Product ID` can legitimately repeat when the same product ships on
different terms within an order. **Deduplicating on those keys would destroy real revenue.**

In [3]:
exact = int(df.duplicated().sum())
excl_rowid = int(df.drop(columns=["Row ID"]).duplicated().sum())
order_product = int(df.duplicated(subset=["Order ID", "Product ID"]).sum())

print(f"Exact duplicate rows (all 21 columns)   : {exact}")
print(f"Duplicates ignoring the Row ID artifact : {excl_rowid}")
print(f"Repeated (Order ID, Product ID) pairs   : {order_product}")
print()

if order_product:
    key = df.duplicated(subset=["Order ID", "Product ID"], keep=False)
    sample = (df.loc[key, ["Order ID", "Product ID", "Sales", "Quantity", "Discount", "Profit"]]
                .sort_values(["Order ID", "Product ID"]).head(6))
    print("Inspecting the repeats — differing Sales/Quantity/Discount means these are DISTINCT")
    print("commercial lines, not data errors:")
    print(sample.to_string(index=False))

print()
print("DECISION: remove nothing. Exact duplicates are 0; (Order ID, Product ID) repeats carry")
print("different economics per line and are legitimate multi-line orders.")

record("Duplicate handling", len(df), len(df),
       f"0 exact duplicates. {order_product} (Order ID, Product ID) repeats retained — "
       "distinct commercial lines, not errors.")

Exact duplicate rows (all 21 columns)   : 0
Duplicates ignoring the Row ID artifact : 1
Repeated (Order ID, Product ID) pairs   : 8

Inspecting the repeats — differing Sales/Quantity/Discount means these are DISTINCT
commercial lines, not data errors:
      Order ID      Product ID  Sales  Quantity  Discount   Profit
CA-2015-103135 OFF-BI-10000069 135.09         9       0.0  62.1414
CA-2015-103135 OFF-BI-10000069  90.06         6       0.0  41.4276
CA-2016-129714 OFF-PA-10001970  24.56         2       0.0  11.5432
CA-2016-129714 OFF-PA-10001970  49.12         4       0.0  23.0864
CA-2016-137043 FUR-FU-10003664 572.76         6       0.0 166.1004
CA-2016-137043 FUR-FU-10003664 286.38         3       0.0  83.0502

DECISION: remove nothing. Exact duplicates are 0; (Order ID, Product ID) repeats carry
different economics per line and are legitimate multi-line orders.


## 2.3 Type casting

Types are enforced early to surface silent coercion. Two decisions matter:

- **`Postal Code` becomes a string.** It arrived as an integer, which has already destroyed
  leading zeros for New England ZIPs. It is a label, not a quantity — nobody computes a mean
  postal code. We zero-pad to restore the 5-digit form.
- **Dates are parsed with an explicit format** (`%m/%d/%Y`). Letting pandas infer per-row invites
  silent day/month transposition on ambiguous values like `3/4/2016`.

In [4]:
before_dtypes = df.dtypes.astype(str).copy()

df["Order Date"] = pd.to_datetime(df["Order Date"], format="%m/%d/%Y")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], format="%m/%d/%Y")

zeros_restored = int((df["Postal Code"].astype(str).str.len() < 5).sum())
df["Postal Code"] = df["Postal Code"].astype(str).str.zfill(5)

for col in ["Ship Mode", "Segment", "Region", "Category", "Sub-Category"]:
    df[col] = df[col].astype("category")

df["Sales"] = df["Sales"].astype("float64")
df["Profit"] = df["Profit"].astype("float64")
df["Discount"] = df["Discount"].astype("float64")
df["Quantity"] = df["Quantity"].astype("int16")

print(f"Postal codes with restored leading zeros: {zeros_restored:,}")
print()
comparison = pd.DataFrame({"before": before_dtypes, "after": df.dtypes.astype(str)})
print(comparison[comparison["before"] != comparison["after"]].to_string())

record("Type casting", len(df), len(df),
       f"Dates -> datetime64 (explicit %m/%d/%Y). 5 columns -> category. Postal Code -> "
       f"zero-padded string, restoring leading zeros on {zeros_restored:,} rows.")

Postal codes with restored leading zeros: 449

             before           after
Order Date      str  datetime64[us]
Ship Date       str  datetime64[us]
Ship Mode       str        category
Segment         str        category
Postal Code   int64             str
Region          str        category
Category        str        category
Sub-Category    str        category
Quantity      int64           int16


## 2.4 Business-rule validation

Domain assertions. Violations are **logged, not silently dropped** (`STRUCTURE.md` §2).

In [5]:
rules = {
    "Sales > 0": df["Sales"] > 0,
    "Quantity >= 1": df["Quantity"] >= 1,
    "Discount within [0, 1]": df["Discount"].between(0, 1),
    "Ship Date >= Order Date": df["Ship Date"] >= df["Order Date"],
    "Order Date within 2014-2017": df["Order Date"].between("2014-01-01", "2017-12-31"),
    "Fulfilment lag <= 14 days": (df["Ship Date"] - df["Order Date"]).dt.days <= 14,
    "Profit magnitude <= Sales x 10": df["Profit"].abs() <= df["Sales"] * 10,
}

rule_report = pd.DataFrame([
    {"rule": name, "violations": int((~mask).sum()),
     "result": "PASS" if (~mask).sum() == 0 else "REVIEW"}
    for name, mask in rules.items()
])
print(rule_report.to_string(index=False))

violations = rule_report.loc[rule_report["result"] == "REVIEW", "rule"].tolist()
print()
print("All business rules pass." if not violations else f"Rules needing review: {violations}")

record("Business-rule validation", len(df), len(df),
       f"{len(rules)} rules tested, {rule_report['violations'].sum()} total violations. "
       "No rows removed.")

                          rule  violations result
                     Sales > 0           0   PASS
                 Quantity >= 1           0   PASS
        Discount within [0, 1]           0   PASS
       Ship Date >= Order Date           0   PASS
   Order Date within 2014-2017           0   PASS
     Fulfilment lag <= 14 days           0   PASS
Profit magnitude <= Sales x 10           0   PASS

All business rules pass.


## 2.5 Outliers — deliberately retained

The `-$6,600` line is not a data error. **It is the finding.** Removing extreme values here would
delete precisely the loss concentration the whole analysis exists to locate.

We quantify the tail so it is documented, then leave it alone. Winsorised variants appear later
only as a *robustness check* inside statistical tests — never as the headline.

In [6]:
for col in ["Sales", "Profit"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int((~df[col].between(lo, hi)).sum())
    print(f"{col:<7} IQR fence [{lo:>10,.2f}, {hi:>9,.2f}] -> {n_out:,} points outside "
          f"({n_out / len(df):.1%})")
    print(f"        actual range [{df[col].min():,.2f}, {df[col].max():,.2f}], "
          f"mean {df[col].mean():,.2f} vs median {df[col].median():,.2f}")

print()
print("Both fields are heavily right-skewed (mean >> median), so mean-only reporting would")
print("misrepresent the typical line. Median + IQR is reported alongside the mean throughout.")
print("DECISION: retain all outliers. Documented, not deleted.")

record("Outlier treatment", len(df), len(df),
       "No outliers removed. Extreme losses are the analytic target, not contamination.")

Sales   IQR fence [   -271.71,    498.93] -> 1,167 points outside (11.7%)
        actual range [0.44, 22,638.48], mean 229.86 vs median 54.49
Profit  IQR fence [    -39.72,     70.82] -> 1,881 points outside (18.8%)
        actual range [-6,599.98, 8,399.98], mean 28.66 vs median 8.67

Both fields are heavily right-skewed (mean >> median), so mean-only reporting would
misrepresent the typical line. Median + IQR is reported alongside the mean throughout.
DECISION: retain all outliers. Documented, not deleted.


## 2.6 Column removal

Three columns go, each for a distinct reason.

In [7]:
drops = {
    "Row ID": "Surrogate index artifact — carries no information and invites accidental use as a feature.",
    "Country": f"Zero variance (all '{df['Country'].iloc[0]}') — cannot discriminate anything.",
    "Customer Name": "QUASI-PII. Data minimisation: Customer ID supports every customer feature we need.",
}
for col, why in drops.items():
    print(f"DROP {col:<15} {why}")

df = df.drop(columns=list(drops))
record("Column removal", len(df), len(df),
       "Dropped Row ID (artifact), Country (zero variance), Customer Name (PII minimisation).")

DROP Row ID          Surrogate index artifact — carries no information and invites accidental use as a feature.
DROP Country         Zero variance (all 'United States') — cannot discriminate anything.
DROP Customer Name   QUASI-PII. Data minimisation: Customer ID supports every customer feature we need.


## 2.7 Derived columns

Per `DOCS/Superstore_Dataset_Analysis.md` §7. Two carry leakage risk and are handled explicitly:

- **`profit_margin`** is a deterministic function of `Profit`. It is an *analysis* variable and is
  asserted **out** of the Stage 5b feature matrix — a model predicting `is_loss` from margin would
  score a perfect AUC and mean nothing.
- **Customer features are computed as-of**, using only orders strictly *before* the current one.
  A naive `groupby.transform('count')` would leak the customer's entire future into every row.

In [8]:
df["profit_margin"] = df["Profit"] / df["Sales"]
df["is_loss"] = (df["Profit"] < 0).astype("int8")
df["fulfillment_lag"] = (df["Ship Date"] - df["Order Date"]).dt.days.astype("int16")
df["unit_price"] = df["Sales"] / df["Quantity"]

df["discount_band"] = pd.cut(
    df["Discount"], bins=[-0.001, 0.0001, 0.20, 0.50, 1.0],
    labels=["0%", "1-20%", "21-50%", "51%+"],
).astype(pd.CategoricalDtype(["0%", "1-20%", "21-50%", "51%+"], ordered=True))

df["order_year"] = df["Order Date"].dt.year.astype("int16")
df["order_quarter"] = df["Order Date"].dt.quarter.astype("int8")
df["order_month"] = df["Order Date"].dt.month.astype("int8")
df["order_month_start"] = df["Order Date"].values.astype("datetime64[M]")
df["order_line_count"] = df.groupby("Order ID")["Order ID"].transform("size").astype("int16")

# --- Leakage-safe customer features -------------------------------------------------
df = df.sort_values(["Customer ID", "Order Date"], kind="stable").reset_index(drop=True)
order_level = df[["Customer ID", "Order ID", "Order Date"]].drop_duplicates("Order ID")
order_level = order_level.sort_values(["Customer ID", "Order Date"], kind="stable")
order_level["cust_prior_orders"] = order_level.groupby("Customer ID").cumcount().astype("int16")
order_level["cust_days_since_prior"] = (
    order_level.groupby("Customer ID")["Order Date"].diff().dt.days.fillna(-1).astype("int32")
)
df = df.merge(
    order_level[["Order ID", "cust_prior_orders", "cust_days_since_prior"]],
    on="Order ID", how="left", validate="many_to_one",
)

print("Derived columns:")
for c in ["profit_margin", "is_loss", "fulfillment_lag", "unit_price", "discount_band",
          "order_year", "order_quarter", "order_month", "order_line_count",
          "cust_prior_orders", "cust_days_since_prior"]:
    print(f"  {c}")

Derived columns:
  profit_margin
  is_loss
  fulfillment_lag
  unit_price
  discount_band
  order_year
  order_quarter
  order_month
  order_line_count
  cust_prior_orders
  cust_days_since_prior


In [9]:
# Leakage assertion: the first order of every customer must show zero prior orders.
first_orders = df.loc[df.groupby("Customer ID")["Order Date"].idxmin(), "cust_prior_orders"]
assert (first_orders == 0).all(), "LEAKAGE: a customer's first order shows prior history."
assert df["cust_prior_orders"].min() == 0
print("Leakage assertion passed: customer features are strictly backward-looking.")
print(f"  cust_prior_orders range: {df['cust_prior_orders'].min()} to {df['cust_prior_orders'].max()}")

record("Feature derivation", len(df), len(df),
       "Added 11 derived columns. Customer features computed as-of (backward-looking only) and "
       "asserted leakage-free.")

Leakage assertion passed: customer features are strictly backward-looking.
  cust_prior_orders range: 0 to 16


## 2.8 Post-derivation sanity checks

Derived columns get the same validation discipline as source columns.

In [10]:
post = {
    "profit_margin finite": np.isfinite(df["profit_margin"]).all(),
    "profit_margin <= 1": bool((df["profit_margin"] <= 1.0000001).all()),
    "is_loss agrees with Profit sign": bool((df["is_loss"] == (df["Profit"] < 0)).all()),
    "fulfillment_lag >= 0": bool((df["fulfillment_lag"] >= 0).all()),
    "unit_price > 0": bool((df["unit_price"] > 0).all()),
    "discount_band has no nulls": bool(df["discount_band"].notna().all()),
    "order_line_count >= 1": bool((df["order_line_count"] >= 1).all()),
    "row count unchanged": len(df) == n_start,
}
for name, ok in post.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")
assert all(post.values()), "Post-derivation validation failed."

print()
print(f"Margin range: [{df['profit_margin'].min():.2%}, {df['profit_margin'].max():.2%}] — the")
print("floor below -100% is real: a discounted line can lose more than its own revenue.")

  PASS  profit_margin finite
  PASS  profit_margin <= 1
  PASS  is_loss agrees with Profit sign
  PASS  fulfillment_lag >= 0
  PASS  unit_price > 0
  PASS  discount_band has no nulls
  PASS  order_line_count >= 1
  PASS  row count unchanged

Margin range: [-275.00%, 50.00%] — the
floor below -100% is real: a discounted line can lose more than its own revenue.


## 2.9 Cleaning log

In [11]:
cleaning_log = pd.DataFrame(log)
print(cleaning_log[["step", "rows_before", "rows_after", "rows_affected"]].to_string(index=False))
print()
print(f"Rows in : {n_start:,}")
print(f"Rows out: {len(df):,}  (zero rows removed — every exclusion decision was a deliberate 'no')")
cleaning_log

                    step  rows_before  rows_after  rows_affected
  Missing-value handling         9994        9994              0
      Duplicate handling         9994        9994              0
            Type casting         9994        9994              0
Business-rule validation         9994        9994              0
       Outlier treatment         9994        9994              0
          Column removal         9994        9994              0
      Feature derivation         9994        9994              0

Rows in : 9,994
Rows out: 9,994  (zero rows removed — every exclusion decision was a deliberate 'no')


,step,rows_before,rows_after,rows_affected,rationale
0,Missing-value handling,9994,9994,0,"No nulls, blanks, or sentinels found. No imput..."
1,Duplicate handling,9994,9994,0,"0 exact duplicates. 8 (Order ID, Product ID) r..."
2,Type casting,9994,9994,0,Dates -> datetime64 (explicit %m/%d/%Y). 5 col...
3,Business-rule validation,9994,9994,0,"7 rules tested, 0 total violations. No rows re..."
4,Outlier treatment,9994,9994,0,No outliers removed. Extreme losses are the an...
5,Column removal,9994,9994,0,"Dropped Row ID (artifact), Country (zero varia..."
6,Feature derivation,9994,9994,0,Added 11 derived columns. Customer features co...


## 2.10 Persist

In [12]:
OUT = PROCESSED / "superstore_clean.parquet"
df.to_parquet(OUT, index=False)
cleaning_log.to_csv(PROCESSED / "_cleaning_log.csv", index=False)

print(f"Written: data/processed/{OUT.name}  ({OUT.stat().st_size / 1_048_576:.2f} MB)")
print(f"Shape  : {df.shape[0]:,} rows x {df.shape[1]} columns")
print()
print("Lineage: Superstore.csv -> data/raw/ -> data/processed/superstore_clean.parquet")
df.head(3)

Written: data/processed/superstore_clean.parquet  (0.45 MB)
Shape  : 9,994 rows x 30 columns

Lineage: Superstore.csv -> data/raw/ -> data/processed/superstore_clean.parquet


,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,profit_margin,is_loss,fulfillment_lag,unit_price,discount_band,order_year,order_quarter,order_month,order_month_start,order_line_count,cust_prior_orders,cust_days_since_prior
0,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Consumer,San Francisco,California,94122,West,OFF-BI-10004390,Office Supplies,Binders,GBC DocuBind 200 Manual Binding Machine,673.568,2,0.2,252.5880,0.375,0,5,336.784,1-20%,2014,1,3,2014-03-01,2,0,-1
1,CA-2014-128055,2014-03-31,2014-04-05,Standard Class,AA-10315,Consumer,San Francisco,California,94122,West,OFF-AP-10002765,Office Supplies,Appliances,Fellowes Advanced Computer Series Surge Protec...,52.980,2,0.0,14.8344,0.280,0,5,26.490,0%,2014,1,3,2014-03-01,2,0,-1
2,CA-2014-138100,2014-09-15,2014-09-20,Standard Class,AA-10315,Consumer,New York City,New York,10011,East,OFF-PA-10000349,Office Supplies,Paper,Easy-staple paper,14.940,3,0.0,7.0218,0.470,0,5,4.980,0%,2014,3,9,2014-09-01,2,1,168


## Stage 2 — Gate Checklist

- [x] Missing-value strategy documented per column (verified none, including disguised nulls)
- [x] Duplicates assessed and handled with justification (retained — legitimate multi-line orders)
- [x] All columns have correct dtypes (dates parsed explicitly, Postal Code as zero-padded string)
- [x] Business-rule validations pass — 7 rules, 0 violations
- [x] Cleaning log exists with rows affected and rationale per step
- [x] Cleaned data saved to `data/processed/`
- [x] Leakage assertion on customer features passes

**Next:** `03_eda.ipynb` — hypothesis-driven exploration against the Stage 0 issue tree.